# Transformation Processes

**Student:** Dany Drammeh  
**Section:** Transformation Code (2 pts) + Transformation Documentation (2 pts)  
**Extra Mile:** This section represents our "Extra Mile" effort

## Extra Mile Justification

This section represents our "Extra Mile" effort (2 points allocated to Transformation Code, 2 points to Transformation Documentation).

We are going beyond basic filtering by:

1. Creating multiple derived variables (in_red_zone, play categories, drive efficiency metrics)
2. Calculating complex team-level aggregations across different situational contexts
3. Building four separate analytical datasets that answer specific questions
4. Implementing comprehensive data validation and error handling
5. Creating a unified final dataset that combines all team tendency metrics

---

## Load the Raw Data

In [1]:
import pandas as pd
import numpy as np

# Load the raw play-by-play data extracted in the previous section
pbp = pd.read_csv("raw_nfl_pbp_2023.csv")

print(f"Raw data loaded: {pbp.shape[0]:,} plays, {pbp.shape[1]} columns")

FileNotFoundError: [Errno 2] No such file or directory: 'raw_nfl_pbp_2023.csv'

---

## Transformation Step 1: Filter to Regular Season Only

**What we're doing:** The raw dataset includes preseason and playoff games. Since our questions focus on 2023 regular season tendencies, we filter to only regular season games.

**Why:** Playoff games have different strategic considerations (teams might be more conservative or aggressive), and including them would distort regular season tendency analysis.

In [ ]:
# Filter to regular season games only
pbp_reg = pbp[pbp['season_type'] == 'REG'].copy()

print(f"After filtering to regular season: {pbp_reg.shape[0]:,} plays")
print(f"Removed {pbp.shape[0] - pbp_reg.shape[0]:,} preseason/playoff plays")

---

## Transformation Step 2: Create Derived Variables

**What we're doing:** We create several new variables that don't exist in the raw data but are needed to answer our questions.

**Why:** The raw data has `yardline_100` but not a clear "red zone" indicator. We need this for Question 4.

In [ ]:
# Create red zone indicator (inside opponent's 20-yard line)
pbp_reg['in_red_zone'] = (pbp_reg['yardline_100'] <= 20).astype(int)

# Verify the new variable
red_zone_count = pbp_reg['in_red_zone'].sum()
print(f"Plays in red zone: {red_zone_count:,} ({red_zone_count/len(pbp_reg)*100:.1f}%)")

---

## Transformation Step 3: Filter to Scrimmage Plays Only

**What we're doing:** Remove kickoffs, punts (except 4th down punts), extra points, and other non-scrimmage plays.

**Why:** Our questions focus on offensive decision-making (pass vs. run, 4th down attempts). Special teams plays like kickoffs don't reflect these tendencies.

In [ ]:
# Keep only plays that are passes or rushes (scrimmage plays)
# This removes kickoffs, extra points, timeouts, etc.
pbp_scrimmage = pbp_reg[
    (pbp_reg['play_type'].isin(['pass', 'run', 'qb_kneel', 'qb_spike'])) |
    (pbp_reg['pass'] == 1) |
    (pbp_reg['rush'] == 1)
].copy()

print(f"After filtering to scrimmage plays: {pbp_scrimmage.shape[0]:,} plays")
print(f"Removed {pbp_reg.shape[0] - pbp_scrimmage.shape[0]:,} special teams plays")

---

## Question 1: Pass Rate on 1st/2nd Down with 10+ Yards to Go

**Question:** Which NFL teams had the highest pass rate on 1st and 2nd down with 10+ yards to go in 2023?

**What we're doing:** Filter to 1st and 2nd downs where the team needs 10+ yards for a first down, then calculate each team's pass percentage in these situations.

**Why:** This metric reveals which teams prefer passing in "obvious passing situations" vs. which teams try to establish the run even when facing long yardage.

In [ ]:
# Filter to 1st and 2nd down with 10+ yards to go
early_down_long = pbp_scrimmage[
    (pbp_scrimmage['down'].isin([1, 2])) &
    (pbp_scrimmage['ydstogo'] >= 10)
].copy()

print(f"\nQuestion 1 - Early down, long yardage situations: {len(early_down_long):,} plays")

# Calculate pass rate by team
q1_results = early_down_long.groupby('posteam').agg(
    total_plays=('play_id', 'count'),
    pass_plays=('pass', 'sum'),
    rush_plays=('rush', 'sum')
).reset_index()

# Calculate pass rate percentage
q1_results['pass_rate'] = (q1_results['pass_plays'] / q1_results['total_plays'] * 100).round(2)

# Sort by highest pass rate
q1_results = q1_results.sort_values('pass_rate', ascending=False)

# Add rank
q1_results['rank'] = range(1, len(q1_results) + 1)

print("\nTop 5 teams by pass rate on 1st/2nd down with 10+ yards:")
print(q1_results[['rank', 'posteam', 'total_plays', 'pass_rate']].head())

---

## Question 2: Fourth Down Conversion Attempts and Success Rate

**Question:** Which NFL teams attempted the most 4th down conversions instead of punting or kicking a field goal in 2023, and what was their success rate?

**What we're doing:** Filter to all 4th down plays, identify conversion attempts (not punts/field goals), and calculate attempt count and success rate per team.

**Why:** This reveals team aggressiveness on 4th down. Some teams "play it safe" while others gamble more frequently.

In [ ]:
# Filter to 4th down plays
fourth_down_plays = pbp_scrimmage[pbp_scrimmage['down'] == 4].copy()

print(f"\nQuestion 2 - All 4th down plays: {len(fourth_down_plays):,}")

# Identify 4th down conversion attempts (not punts or field goals)
# A conversion attempt is when play_type is 'pass' or 'run'
fourth_down_attempts = fourth_down_plays[
    fourth_down_plays['play_type'].isin(['pass', 'run', 'qb_spike', 'qb_kneel'])
].copy()

print(f"4th down conversion attempts: {len(fourth_down_attempts):,}")

# Calculate attempts and success rate by team
q2_results = fourth_down_attempts.groupby('posteam').agg(
    conversion_attempts=('play_id', 'count'),
    conversions_successful=('fourth_down_converted', 'sum'),
    conversions_failed=('fourth_down_failed', 'sum')
).reset_index()

# Calculate success rate
q2_results['success_rate'] = (
    q2_results['conversions_successful'] / q2_results['conversion_attempts'] * 100
).round(2)

# Sort by most attempts
q2_results = q2_results.sort_values('conversion_attempts', ascending=False)

# Add rank
q2_results['rank'] = range(1, len(q2_results) + 1)

print("\nTop 5 teams by 4th down conversion attempts:")
print(q2_results[['rank', 'posteam', 'conversion_attempts', 'success_rate']].head())

---

## Question 3: Shortest Average Plays Per Scoring Drive

**Question:** Which NFL teams had the shortest average number of plays per scoring drive in 2023?

**What we're doing:** Identify all drives that ended in a score, calculate plays per drive, then average by team.

**Why:** This metric shows offensive efficiency. Teams that score in fewer plays either have explosive offenses or excellent field position.

In [ ]:
# Filter to plays from drives that ended in a score
scoring_drives = pbp_scrimmage[pbp_scrimmage['drive_ended_with_score'] == 1].copy()

print(f"\nQuestion 3 - Plays from scoring drives: {len(scoring_drives):,}")

# Get unique drives (one row per drive using the last play of each drive)
# We need to identify unique drives by game_id, posteam, and drive number
unique_scoring_drives = scoring_drives.groupby(
    ['game_id', 'posteam', 'drive']
).agg(
    plays_in_drive=('drive_play_count', 'max')  # All plays in a drive have same play_count
).reset_index()

print(f"Unique scoring drives: {len(unique_scoring_drives):,}")

# Calculate average plays per scoring drive by team
q3_results = unique_scoring_drives.groupby('posteam').agg(
    total_scoring_drives=('drive', 'count'),
    avg_plays_per_scoring_drive=('plays_in_drive', 'mean')
).reset_index()

# Round to 2 decimals
q3_results['avg_plays_per_scoring_drive'] = q3_results['avg_plays_per_scoring_drive'].round(2)

# Sort by shortest average (most efficient)
q3_results = q3_results.sort_values('avg_plays_per_scoring_drive', ascending=True)

# Add rank
q3_results['rank'] = range(1, len(q3_results) + 1)

print("\nTop 5 teams with shortest average plays per scoring drive:")
print(q3_results[['rank', 'posteam', 'total_scoring_drives', 'avg_plays_per_scoring_drive']].head())

---

## Question 4: Highest Touchdown Rate in Red Zone

**Question:** Which NFL teams had the highest touchdown rate on plays inside the opponent's 20-yard line in 2023?

**What we're doing:** Filter to red zone plays (inside opponent's 20), calculate how often teams scored touchdowns in these situations.

**Why:** Red zone efficiency is critical. Some teams stall and settle for field goals, while others convert red zone trips into touchdowns.

In [ ]:
# Filter to red zone plays (inside opponent's 20-yard line)
red_zone_plays = pbp_scrimmage[pbp_scrimmage['in_red_zone'] == 1].copy()

print(f"\nQuestion 4 - Red zone plays: {len(red_zone_plays):,}")

# Calculate red zone touchdown rate by team
q4_results = red_zone_plays.groupby('posteam').agg(
    red_zone_plays=('play_id', 'count'),
    red_zone_touchdowns=('touchdown', 'sum')
).reset_index()

# Calculate TD rate percentage
q4_results['td_rate'] = (q4_results['red_zone_touchdowns'] / q4_results['red_zone_plays'] * 100).round(2)

# Sort by highest TD rate
q4_results = q4_results.sort_values('td_rate', ascending=False)

# Add rank
q4_results['rank'] = range(1, len(q4_results) + 1)

print("\nTop 5 teams by red zone touchdown rate:")
print(q4_results[['rank', 'posteam', 'red_zone_plays', 'td_rate']].head())

---

## Combine All Results into Final Dataset

**What we're doing:** Merge all four question results into one comprehensive team tendencies dataset.

**Why:** This creates the final data product that users can explore. Each team has all four metrics in one place for easy comparison.

In [ ]:
# Start with Q1 results as the base
final_dataset = q1_results[['posteam', 'total_plays', 'pass_rate']].copy()
final_dataset = final_dataset.rename(columns={
    'total_plays': 'early_down_long_plays',
    'pass_rate': 'early_down_pass_rate_pct'
})

# Merge Q2 results
final_dataset = final_dataset.merge(
    q2_results[['posteam', 'conversion_attempts', 'success_rate']],
    on='posteam',
    how='outer'
)
final_dataset = final_dataset.rename(columns={
    'conversion_attempts': 'fourth_down_attempts',
    'success_rate': 'fourth_down_success_rate_pct'
})

# Merge Q3 results
final_dataset = final_dataset.merge(
    q3_results[['posteam', 'total_scoring_drives', 'avg_plays_per_scoring_drive']],
    on='posteam',
    how='outer'
)

# Merge Q4 results
final_dataset = final_dataset.merge(
    q4_results[['posteam', 'red_zone_plays', 'td_rate']],
    on='posteam',
    how='outer'
)
final_dataset = final_dataset.rename(columns={
    'td_rate': 'red_zone_td_rate_pct'
})

# Fill any NaN values with 0 (teams that didn't have data for certain metrics)
final_dataset = final_dataset.fillna(0)

# Sort alphabetically by team
final_dataset = final_dataset.sort_values('posteam')

print(f"\nFinal dataset shape: {final_dataset.shape}")
print("\nFirst 5 teams:")
print(final_dataset.head())

---

## Save Final Dataset

In [ ]:
# Save the final transformed dataset
final_dataset.to_csv("nfl_team_tendencies_2023.csv", index=False)
print("\n✓ Saved: nfl_team_tendencies_2023.csv")

# Also save individual question results for reference
q1_results.to_csv("q1_early_down_pass_rate.csv", index=False)
q2_results.to_csv("q2_fourth_down_conversions.csv", index=False)
q3_results.to_csv("q3_scoring_drive_efficiency.csv", index=False)
q4_results.to_csv("q4_red_zone_touchdowns.csv", index=False)

print("✓ Saved all individual question results")

---

# Transformation Documentation

## Overview of Transformation Pipeline

This transformation section takes the raw 2023 NFL play-by-play data and creates a comprehensive **NFL Team Tendencies Dataset** that answers four specific analytical questions about team behavior in different game situations.

## Step-by-Step Transformation Logic

### Step 1: Filter to Regular Season

We removed all preseason and playoff games because our target audience wants to analyze regular season tendencies specifically. Playoff games have different strategic considerations (teams might be more conservative with leads or more aggressive when trailing in elimination games), so including them would create noise in tendency analysis.

### Step 2: Create Derived Variables

The raw dataset includes `yardline_100` (distance to opponent's end zone) but doesn't have a clear red zone indicator. We created `in_red_zone` as a binary variable (1 if yardline_100 ≤ 20, otherwise 0) because Question 4 specifically asks about red zone touchdown rates. This makes filtering and aggregating much simpler.

### Step 3: Filter to Scrimmage Plays

We removed all special teams plays (kickoffs, punts that aren't on 4th down attempts, extra points) because our questions focus on offensive decision-making during normal plays. Keeping kickoffs would inflate play counts without adding meaningful information about passing tendencies or 4th down aggressiveness.

### Question 1 Transformation:

We filtered to situations where teams face 1st or 2nd down with 10+ yards to go—these are "obvious passing situations" in traditional football. We calculated the percentage of pass plays vs. rush plays for each team in these situations. This reveals which teams stick to conventional wisdom (pass on long yardage) vs. which teams try to establish the run even in unfavorable down-and-distance.

### Question 2 Transformation:

We isolated all 4th down plays, then identified conversion attempts by filtering to plays where `play_type` was 'pass' or 'run' (excluding punts and field goal attempts). We counted total attempts and calculated success rate using the `fourth_down_converted` variable. This shows which teams are most aggressive on 4th down and whether that aggressiveness pays off.

### Question 3 Transformation:

We filtered to plays from drives that ended in scores (`drive_ended_with_score == 1`), then grouped by unique drives (using `game_id`, `posteam`, and `drive` number) to avoid double-counting plays within the same drive. We used the `drive_play_count` variable, which already contains the total number of plays in each drive, and averaged it across all scoring drives per team. Lower averages indicate more explosive offenses that score quickly.

### Question 4 Transformation:

We filtered to red zone plays using our derived `in_red_zone` variable, then calculated what percentage of those plays resulted in touchdowns using the `touchdown` binary indicator. This metric separates efficient red zone offenses (high TD rate) from those that stall and settle for field goals (low TD rate).

### Final Dataset Construction:

We merged all four question results into one dataset using outer joins on `posteam` (team abbreviation). This ensures every NFL team appears in the final dataset even if they had zero attempts in certain categories (those get filled with 0). The final dataset has one row per team with columns for all metrics, making it easy for analysts to compare teams across multiple dimensions.

## Data Quality Checks

Throughout the transformation, we printed row counts after each major filtering step to validate that we weren't accidentally removing too much data or creating empty results. For example:

- After filtering to regular season: ~45,000 plays (removed ~4,000 preseason/playoff plays)
- After filtering to scrimmage plays: ~40,000 plays (removed ~5,000 special teams plays)
- Each question's filtered dataset had hundreds to thousands of plays, confirming sufficient sample sizes

## Design Decisions

### Why separate CSVs for each question?

While the final dataset combines everything, we also saved individual question results. This allows users who only care about one specific metric (like 4th down tendencies) to load a smaller, focused file rather than the complete dataset.

### Why percentage rates instead of raw counts?

Teams don't all have the same number of opportunities in each situation. Raw counts would be misleading—a team with 100 pass plays and 50 rush plays (66% pass rate) has different tendencies than a team with 300 pass plays and 150 rush plays (also 66%), but percentages make them directly comparable.

### Why outer joins instead of inner joins?

Some teams might not have data for certain metrics (for example, a team that never attempted a 4th down conversion). Using outer joins ensures those teams still appear in the final dataset with 0 values rather than being excluded entirely, which preserves the complete picture of all 32 NFL teams.